In [1]:
import numpy as np

# Configuration parameters for the (3,4)-regular LDPC base matrix
size = 3     # Example Submatrix size (Z value for WiMAX/WiFi sizes vary, 38 is typical for some max shift values here like 37)
M_b = 3       # Number of macro rows in base matrix
N_b = 4      # Number of macro columns in base matrix

# Shift values for the quasi-cyclic matrix (-1 means Bypass/All-Zero submatrix)
# Edited to ensure exactly 1 bypass per macro row and macro column.
# shift_matrix = np.array([
#     [ 29, 30,  0,  8, 33, 22, 17,  4, 27, 28, 20, 27, 24, 23, -1,  0],
#     [ 37, 31, 18, 23, 11, 21,  6, 20, 32,  9, 12, 29, 10,  0, 13, -1],
#     [ 25, 22,  4, 34, 31,  3, 14, 15,  4,  2, 14, 18, 13, -1, 22, 24]
# ])

shift_matrix = np.array([
    [ 1, 0,-1, 2],
    [-1, 2, 0, 0],
    [ 1, 0, 2, 1]
])

M = M_b * size
N = N_b * size

# Generate the full parity-check matrix H
matrix_orig = np.zeros((M, N), dtype=int)

for i in range(M_b):
    for j in range(N_b):
        shift = shift_matrix[i, j]
        row_start = i * size
        col_start = j * size
        
        if shift == -1:
            submat = np.zeros((size, size), dtype=int)
        else:
            submat = np.roll(np.eye(size, dtype=int), shift, axis=1)
            
        matrix_orig[row_start:row_start+size, col_start:col_start+size] = submat

matrix = matrix_orig.copy()
print("Generated Full H Matrix:")
print(matrix_orig)


Generated Full H Matrix:
[[0 1 0 1 0 0 0 0 0 0 0 1]
 [0 0 1 0 1 0 0 0 0 1 0 0]
 [1 0 0 0 0 1 0 0 0 0 1 0]
 [0 0 0 0 0 1 1 0 0 1 0 0]
 [0 0 0 1 0 0 0 1 0 0 1 0]
 [0 0 0 0 1 0 0 0 1 0 0 1]
 [0 1 0 1 0 0 0 0 1 0 1 0]
 [0 0 1 0 1 0 1 0 0 0 0 1]
 [1 0 0 0 0 1 0 1 0 1 0 0]]


In [2]:
rows, cols = matrix_orig.shape
num_layers = rows // size

def prev_one_in_row(row_ones, col):
    before = [x for x in row_ones if x < col]
    if before:
        return max(before)
    else:
        return max(row_ones)

# print("Bypass List (previous 1 to bypass):")
for j in range(cols):
    bypass_val = 15 # Default 4'hF
    for l in range(num_layers):
        sub_col = matrix_orig[l*size:(l+1)*size, j]
        if np.all(sub_col == 0):
            abs_row = (l * size) + (j % size)
            row_ones = np.where(matrix_orig[abs_row, :] == 1)[0].tolist()
            if row_ones:
                bypass_val = prev_one_in_row(row_ones, j)
            break
            
    print(f"        bypass[{j:2d}] = 4'h{bypass_val:X};")


        bypass[ 0] = 4'h9;
        bypass[ 1] = 4'hA;
        bypass[ 2] = 4'hB;
        bypass[ 3] = 4'hF;
        bypass[ 4] = 4'hF;
        bypass[ 5] = 4'hF;
        bypass[ 6] = 4'h3;
        bypass[ 7] = 4'h4;
        bypass[ 8] = 4'h5;
        bypass[ 9] = 4'hF;
        bypass[10] = 4'hF;
        bypass[11] = 4'hF;


In [3]:
cn_pe_table = []
cn_degree = []

# print("cn_pe_table (indices of 1s per CN, excluding bypasses):")
for r in range(rows):
    ones_idx = np.where(matrix_orig[r, :] == 1)[0].tolist()
    cn_pe_table.append(ones_idx)
    cn_degree.append(len(ones_idx))

    layer = r//size

    for i in range(len(cn_pe_table[r])):
        print(f"        cn_pe_table[{r}][{i}] = {ones_idx[i]};")
        print(f"        cn_layer_table[{r}][{i}] = {layer};\n")

    print(f"\n        cn_degree[{r}] = {cn_degree[-1]};\n")


        cn_pe_table[0][0] = 1;
        cn_layer_table[0][0] = 0;

        cn_pe_table[0][1] = 3;
        cn_layer_table[0][1] = 0;

        cn_pe_table[0][2] = 11;
        cn_layer_table[0][2] = 0;


        cn_degree[0] = 3;

        cn_pe_table[1][0] = 2;
        cn_layer_table[1][0] = 0;

        cn_pe_table[1][1] = 4;
        cn_layer_table[1][1] = 0;

        cn_pe_table[1][2] = 9;
        cn_layer_table[1][2] = 0;


        cn_degree[1] = 3;

        cn_pe_table[2][0] = 0;
        cn_layer_table[2][0] = 0;

        cn_pe_table[2][1] = 5;
        cn_layer_table[2][1] = 0;

        cn_pe_table[2][2] = 10;
        cn_layer_table[2][2] = 0;


        cn_degree[2] = 3;

        cn_pe_table[3][0] = 5;
        cn_layer_table[3][0] = 1;

        cn_pe_table[3][1] = 6;
        cn_layer_table[3][1] = 1;

        cn_pe_table[3][2] = 9;
        cn_layer_table[3][2] = 1;


        cn_degree[3] = 3;

        cn_pe_table[4][0] = 3;
        cn_layer_table[4][0] = 1;

        cn_pe_table[4][1] = 

In [4]:
import numpy as np

# Variable size of submatrix (layer = size)
size = 3

# Input matrix
# matrix = np.array([
#     [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1],
#     [0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0],
#     [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0],
#     [0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0],
#     [0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0],
#     [0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1],
#     [0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0],
#     [0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1],
#     [1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0]
# ])

rows, cols = matrix.shape
num_layers = rows // size

# Replace bypass (all-zero) submatrices with identity matrix
for i in range(0, rows, size):
    for j in range(0, cols, size):
        sub = matrix[i:i+size, j:j+size]
        if np.all(sub == 0):
            matrix[i:i+size, j:j+size] = np.eye(sub.shape[0], sub.shape[1])

def prev_one_in_row(row_ones, col):
    """Return the column index of the previous 1 in the row before 'col'.
    If none exists (col is leftmost 1), wrap around to the last 1 in the row."""
    before = [x for x in row_ones if x < col]
    if before:
        return max(before)   # nearest 1 to the left
    else:
        return max(row_ones) # wrap: last 1 in the row

# Generate edgelist:
# For each column, for each layer, find the column index of the previous 1
# in the same row as the current column's 1. Store as packed hex nibbles.
# Nibble[0] (MSN) = layer 0, Nibble[1] = layer 1, Nibble[2] = layer 2, ...
# print("EdgeList (previous 1 per layer, packed as hex):")
for col in range(cols):
    nibbles = []
    for layer in range(num_layers):
        row_start = layer * size
        # Find which row in this layer has a 1 for this column
        rel_rows = np.where(matrix[row_start:row_start+size, col] == 1)[0]
        if len(rel_rows) == 0:
            nibbles.append(0)
            continue
        abs_row = row_start + rel_rows[0]
        # Get all columns in this row that have a 1
        row_ones = np.where(matrix[abs_row, :] == 1)[0].tolist()
        # Find the previous 1 in the row (with circular wrap)
        prev_idx = prev_one_in_row(row_ones, col)
        nibbles.append(prev_idx)
    # Pack nibbles into a single integer (MSN = layer 0)
    val = 0
    for nib in nibbles:
        val = (val << 4) | nib
    bits = num_layers * 4
    print(f"        edgelist[{col:2d}] = {bits}'h{val:0{num_layers}x};")

        edgelist[ 0] = 12'ha99;
        edgelist[ 1] = 12'hbaa;
        edgelist[ 2] = 12'h9bb;
        edgelist[ 3] = 12'h111;
        edgelist[ 4] = 12'h222;
        edgelist[ 5] = 12'h000;
        edgelist[ 6] = 12'h354;
        edgelist[ 7] = 12'h435;
        edgelist[ 8] = 12'h543;
        edgelist[ 9] = 12'h767;
        edgelist[10] = 12'h878;
        edgelist[11] = 12'h686;
